# 05 · The agent — NeMo Agent Toolkit tools

**Purpose** — show the agent answering operational questions, grounded in tool
results, with citations.

**Prerequisites** — 01 run; LLM endpoint reachable. 04 too, for `search_updates`.

### The division of labour
- **Tools compute.** Every number, state, date and count comes from Python.
- **The LLM narrates.** It prioritises and writes prose over what it is given.

It is never asked to count, infer state or recall a fact, so every answer can be
checked against the tool payload that produced it.

In [ ]:
import sys, os, json
sys.path.insert(0, os.path.abspath(".."))          # notebooks/ -> repo root
os.chdir(os.path.abspath(".."))
from dotenv import load_dotenv; load_dotenv()
print("repo root:", os.getcwd())

## The tools

In [ ]:
from app.agent.tools import TOOLS, SPECS
for s in SPECS:
    f = s["function"]
    print(f"  {f['name']:26s} {(f['description'] or '').strip().splitlines()[0]}")

## Routing — deterministic, no LLM required

In [ ]:
from app.agent.agent import plan_keyword
for q in ["What's the status of RO-26-08165?",
          "Which vehicles are unsafe to release?",
          "Give me the afternoon handover",
          "What has EMP014 done this week?",
          "Which jobs will miss their promised time?",
          "Any unusual patterns this week?"]:
    print(f"  {q[:44]:46s} -> {[c['name'] for c in plan_keyword(q)]}")

## Execute — ask the agent

In [ ]:
# "now" follows the newest event in the log; set ASOIA_NOW only to pin one run.
from app.agent.agent import ask
a = ask("Which vehicles cannot be released on safety grounds, and what needs doing?")
print(a.text)
print("\n--- tools used:", [c["name"] for c in a.tool_calls])
print("--- grounded:", a.grounded, "| citations:", len(a.citations))
if a.warnings: print("--- warnings:", a.warnings)

## Insight, not lookup

In [ ]:
a = ask("Are any parts holding up more than one job at once?")
print(a.text)
print("\ncitations:", a.citations[:8])

In [ ]:
a = ask("Give me the afternoon handover, worst first.")
print(a.text)

## NeMo Agent Toolkit

The same tools, registered with NeMo Agent Toolkit. They are written as plain
typed Python functions first so they stay testable, then wrapped without change.
If the toolkit is not installed the agent falls back to its own router, so the
demo never depends on the wrapper.

In [ ]:
from app.agent.nat_functions import NAT_AVAILABLE, REGISTRY
print("aiqtoolkit installed:", NAT_AVAILABLE)
for name, _, schema in REGISTRY:
    print(f"  {name:26s} <- {schema.__name__}")
print()
print(open("app/agent/workflow.yml").read()[:900])

Run the workflow through the toolkit's own CLI:

```bash
uv pip install --python .venv aiqtoolkit
aiq run --config_file app/agent/workflow.yml \
        --input "Which vehicles are unsafe to release?"
```

## Grounding is checked, not assumed

Every figure and repair order in the answer is verified against the tool payload.

In [ ]:
from app.agent.agent import check_grounding
results = [{"tool": "get_ro_state", "result": {"ro_number": "RO-26-08165", "hours_booked": 1.9}}]
for txt in ["RO-26-08165 has 1.9 hours booked.",
            "RO-26-08165 has 7.4 hours booked.",
            "RO-26-09999 is ready for collection."]:
    print(f"  {txt:46s} -> {check_grounding(txt, results) or 'GROUNDED'}")